# Calculate the Main Database Dependency Index (DDI) Results

This notebook calculates behavior-specific DDI values, the behavior-weighted overall DDI, and country-level domestic and external dependency from the finalized database-use records.

## Method

Repeated corresponding-author country entries are retained because they represent distinct affiliation-address entries. Each source-country entry receives a weight of `1 / number of address entries` for that record. When a database has multiple governing countries, its contribution is divided equally among the distinct governing countries. The overall analysis applies behavior weights of Submit = 3, Reuse = 2, and Cite = 1. Behavior-specific DDI values are calculated separately without cross-behavior weights. For each source country, DDI values across governing countries sum to one. External dependency is calculated as `1 - domestic DDI`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Use the specified source and destination files, independent of the notebook working directory.
# Locate the project root whether the notebook is opened from the repository root or the code folder.
project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent
if not (project_root / "data").exists():
    raise FileNotFoundError("Could not locate the repository data folder.")

input_path = project_root / "data" / "01_model_output_records" / "final_database_use_records.xlsx"
output_path = project_root / "data" / "03_ddi_results" / "DDI_main_results.xlsx"
behavior_weights = {"submit": 3, "reuse": 2, "cite": 1}
required_columns = [
    "DOI", "standardized_database_name", "database_use_behavior",
    "database_governing_countries", "corresponding_author_countries"
]
records = pd.read_excel(input_path)
missing_columns = sorted(set(required_columns) - set(records.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if records[required_columns].isna().any().any():
    raise ValueError("Required input fields contain missing values.")

records["database_use_behavior"] = records["database_use_behavior"].astype(str).str.strip().str.lower()
unexpected_behaviors = set(records["database_use_behavior"]) - set(behavior_weights)
if unexpected_behaviors:
    raise ValueError(f"Unexpected behavior labels: {sorted(unexpected_behaviors)}")

print(f"Input records: {len(records):,}")
print("Behavior counts:")
print(records["database_use_behavior"].value_counts().reindex(["submit", "reuse", "cite"], fill_value=0))


Input records: 42,047
Behavior counts:
database_use_behavior
submit    25154
reuse       400
cite      16493
Name: count, dtype: int64


In [2]:
def split_country_entries(value):
    # Split a semicolon-delimited country field and remove blank entries.
    return [part.strip() for part in str(value).split(";") if part.strip()]

expanded_rows = []
for row in records.itertuples(index=False):
    source_entries = split_country_entries(row.corresponding_author_countries)
    # Preserve repeated source-country entries because they represent address entries.
    governing_countries = list(dict.fromkeys(split_country_entries(row.database_governing_countries)))
    if not source_entries or not governing_countries:
        continue
    source_weight = 1.0 / len(source_entries)
    governing_weight = 1.0 / len(governing_countries)
    for source_country in source_entries:
        for governing_country in governing_countries:
            expanded_rows.append({
                "来源国": source_country,
                "数据库治理国": governing_country,
                "使用行为": row.database_use_behavior,
                "record_weight": source_weight * governing_weight,
            })

expanded = pd.DataFrame(expanded_rows)
if expanded.empty:
    raise ValueError("No country-to-governance contributions were generated.")
print(f"Expanded contributions: {len(expanded):,}")
print(f"Source countries: {expanded['来源国'].nunique():,}; governance countries: {expanded['数据库治理国'].nunique():,}")


Expanded contributions: 157,469
Source countries: 77; governance countries: 74


In [3]:
def calculate_ddi(edge_data, weight_column, behavior_label=None):
    # Aggregate weighted use for each source-country/governance-country pair.
    numerator = (
        edge_data.groupby(["来源国", "数据库治理国"], as_index=False)[weight_column]
        .sum().rename(columns={weight_column: "U_ij"})
    )
    # The denominator is the source country's total weighted use across all governance countries.
    denominator = (
        edge_data.groupby("来源国", as_index=False)[weight_column]
        .sum().rename(columns={weight_column: "U_i"})
    )
    result = numerator.merge(denominator, on="来源国", how="left")
    result["DDI"] = result["U_ij"] / result["U_i"]
    if behavior_label is not None:
        result["使用行为"] = behavior_label
    return result.sort_values(["来源国", "DDI"], ascending=[True, False]).reset_index(drop=True)

# Calculate the behavior-weighted overall DDI.
weighted = expanded.copy()
weighted["weighted_contribution"] = weighted["record_weight"] * weighted["使用行为"].map(behavior_weights)
overall_weighted = calculate_ddi(weighted, "weighted_contribution")

# Calculate behavior-specific DDI without cross-behavior weights.
behavior_results = {}
for behavior in ["submit", "cite", "reuse"]:
    behavior_edges = expanded.loc[expanded["使用行为"] == behavior].copy()
    behavior_results[behavior] = calculate_ddi(behavior_edges, "record_weight", behavior_label=behavior)

# Calculate domestic and external dependency from the weighted overall DDI.
domestic = overall_weighted.loc[
    overall_weighted["来源国"] == overall_weighted["数据库治理国"], ["来源国", "DDI"]
].rename(columns={"DDI": "本国依赖DDI"})
country_summary = (
    overall_weighted[["来源国", "U_i"]].drop_duplicates()
    .merge(domestic, on="来源国", how="left")
    .fillna({"本国依赖DDI": 0.0})
)
country_summary["外部依赖度"] = 1.0 - country_summary["本国依赖DDI"]
country_summary = country_summary.sort_values("外部依赖度", ascending=False).reset_index(drop=True)

print("DDI pair counts:")
print({"overall_weighted": len(overall_weighted), **{name: len(result) for name, result in behavior_results.items()}})
print(f"Country summary rows: {len(country_summary)}")


DDI pair counts:
{'overall_weighted': 1183, 'submit': 679, 'cite': 1038, 'reuse': 102}
Country summary rows: 77


In [4]:
# Check that each source country's DDI distribution sums to one.
def validate_row_sums(result, label):
    sums = result.groupby("来源国")["DDI"].sum()
    max_error = float((sums - 1.0).abs().max())
    if max_error > 1e-10:
        raise AssertionError(f"{label}: DDI row sums deviate from one; max error={max_error}")
    return max_error

row_sum_errors = {"overall_weighted": validate_row_sums(overall_weighted, "overall_weighted")}
row_sum_errors.update({name: validate_row_sums(result, name) for name, result in behavior_results.items()})
if not country_summary["外部依赖度"].between(0, 1).all():
    raise AssertionError("External dependency values fall outside [0, 1].")
if not np.isclose(country_summary["本国依赖DDI"] + country_summary["外部依赖度"], 1.0).all():
    raise AssertionError("Domestic DDI and external dependency do not sum to one.")
print("Maximum DDI row-sum errors:", row_sum_errors)

def output_columns(result):
    columns = ["来源国", "数据库治理国", "U_ij", "U_i", "DDI"]
    if "使用行为" in result.columns:
        columns.append("使用行为")
    return result[columns].copy()

output_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    output_columns(overall_weighted).to_excel(writer, sheet_name="overall_weighted", index=False)
    for behavior in ["submit", "cite", "reuse"]:
        output_columns(behavior_results[behavior]).to_excel(writer, sheet_name=behavior, index=False)
    country_summary[["来源国", "U_i", "本国依赖DDI", "外部依赖度"]].to_excel(
        writer, sheet_name="overall_country_summary", index=False
    )


Maximum DDI row-sum errors: {'overall_weighted': 2.220446049250313e-16, 'submit': 1.1102230246251565e-16, 'cite': 2.220446049250313e-16, 'reuse': 1.1102230246251565e-16}


## Output sheets

- `overall_weighted`: Main DDI using Submit/Reuse/Cite weights of 3/2/1.
- `submit`, `cite`, and `reuse`: Behavior-specific DDI distributions.
- `overall_country_summary`: Weighted use volume, domestic DDI, and external dependency for each source country.
